# Kaggle-Ready Multi-Model Multi-Dataset Depth Benchmark (Inference Only)

This notebook compares **your EagleVision adapted checkpoint** against multiple depth estimators.

## Guarantees
- No training.
- Inference-only.
- Kaggle-ready paths/config.
- Fast defaults to keep runtime manageable.
- Separate quantitative and plot outputs **per baseline model vs our model**.

In [ ]:
# Optional (if Kaggle environment misses packages)
# !pip -q install -U transformers datasets pandas matplotlib pillow tqdm scipy

import os
import sys
import math
import random
from pathlib import Path
from typing import Any

import numpy as np
import pandas as pd
from PIL import Image
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn.functional as F

from datasets import load_dataset
from transformers import pipeline

In [ ]:
# -----------------------------
# Kaggle-first configuration
# -----------------------------
CONFIG = {
    "seed": 7,
    "fast_mode": True,
    "max_samples_per_dataset_fast": 40,
    "max_samples_per_dataset_full": 150,
    "num_qualitative_pairs": 10,
    "output_dir": "outputs/kaggle_multi_depth_benchmark",

    # Your adapted checkpoint uploaded as Kaggle input.
    # Example: /kaggle/input/my-eaglevision-checkpoint/best_100ep.pt
    "adapted_checkpoint_path": "/kaggle/input/YOUR_CHECKPOINT_DATASET_NAME/best_100ep.pt",

    # DAV2 backbone checkpoint path. If absent and internet is on, notebook will try downloading.
    "dav2_checkpoint_path": "baseline/depth_anything_v2/checkpoints/depth_anything_v2_metric_hypersim_vits.pth",
    "allow_download_if_missing": True,

    # EagleVision model settings
    "depth_mode": "metric",
    "encoder": "vits",
    "profile": "hypersim",
    "adapter_hidden_channels": 32,
    "normalize_backbone_input": False,

    # Datasets (enable what you have available)
    "datasets": [
        {
            "name": "nyu_depth_v2_hf",
            "type": "hf_nyu",
            "split": "validation",
            "enabled": True,
        },
        {
            "name": "kaggle_scannet_2d",
            "type": "local_scannet_style",
            "root": "/kaggle/input/datasets/klein2111/scannet-2d/scannet_2d",
            "enabled": False,
        },
        {
            "name": "local_rgbd_pairs",
            "type": "folder_pairs",
            "rgb_glob": "data/benchmark_rgbd/rgb/*.png",
            "depth_glob": "data/benchmark_rgbd/depth/*.png",
            "depth_scale": 1000.0,
            "enabled": False,
        },
    ],

    # Model registry.
    # Keep heavy models disabled in fast mode.
    "models": [
        {"id": "ours_adapted", "kind": "eaglevision_adapted", "enabled": True},
        {"id": "ours_dav2_base", "kind": "eaglevision_base", "enabled": True},

        {"id": "depth_anything_v2_small", "kind": "hf_pipeline", "hf_model": "depth-anything/Depth-Anything-V2-Small-hf", "enabled": True},
        {"id": "depth_anything_v1_small", "kind": "hf_pipeline", "hf_model": "LiheYoung/depth-anything-small-hf", "enabled": True},
        {"id": "dpt_large", "kind": "hf_pipeline", "hf_model": "Intel/dpt-large", "enabled": True},
        {"id": "zoedepth_nyu_kitti", "kind": "hf_pipeline", "hf_model": "Intel/zoedepth-nyu-kitti", "enabled": True},

        # Optional heavy baseline
        {"id": "midas_dpt_large", "kind": "torchhub_midas", "model_type": "DPT_Large", "enabled": False},
    ],
}

In [ ]:
# -----------------------------
# Setup repository + imports
# -----------------------------
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("device:", DEVICE)

KAGGLE_WORKING = Path("/kaggle/working")
RUNNING_ON_KAGGLE = KAGGLE_WORKING.exists()

if RUNNING_ON_KAGGLE:
    # Typical Kaggle repo location if notebook is run inside cloned repo
    default_repo = KAGGLE_WORKING / "EagleVision"
    REPO_DIR = default_repo if default_repo.exists() else Path.cwd().resolve()
else:
    REPO_DIR = Path.cwd().resolve()

SRC_DIR = REPO_DIR / "src"
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))
if str(REPO_DIR) not in sys.path:
    sys.path.insert(0, str(REPO_DIR))

from eaglevision.models.depth.depth_anything_wrapper import DepthAnythingWithAdapter
from eaglevision.models.rt_depthnvs import RoundTripDepthNVS
from eaglevision.engine.checkpointing import load_checkpoint

MAX_SAMPLES = CONFIG["max_samples_per_dataset_fast"] if CONFIG["fast_mode"] else CONFIG["max_samples_per_dataset_full"]

random.seed(CONFIG["seed"])
np.random.seed(CONFIG["seed"])
torch.manual_seed(CONFIG["seed"])
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(CONFIG["seed"])

OUT_DIR = REPO_DIR / CONFIG["output_dir"]
for rel in [
    "metrics", "qualitative", "plots/per_model", "tables/per_model",
]:
    (OUT_DIR / rel).mkdir(parents=True, exist_ok=True)

print("repo:", REPO_DIR)
print("output:", OUT_DIR)
print("max samples per dataset:", MAX_SAMPLES)

In [ ]:
# -----------------------------
# Optional DAV2 checkpoint download
# -----------------------------
import subprocess


def ensure_dav2_checkpoint(path_str: str):
    p = Path(path_str)
    if not p.is_absolute():
        p = REPO_DIR / p
    if p.exists():
        print("DAV2 checkpoint found:", p)
        return p

    if not CONFIG["allow_download_if_missing"]:
        raise FileNotFoundError(f"Missing DAV2 checkpoint: {p}")

    print("DAV2 checkpoint missing; attempting download via repo CLI...")
    cmd = [
        sys.executable, "-m", "baseline.depth_anything_v2", "download",
        "--mode", "all", "--profile", CONFIG["profile"], "--encoder", CONFIG["encoder"],
    ]
    result = subprocess.run(cmd, cwd=REPO_DIR, text=True, capture_output=True)
    print(result.stdout[-1500:])
    if result.returncode != 0:
        print(result.stderr[-1500:])
        raise RuntimeError("Failed to download DAV2 checkpoint. Attach it as Kaggle input or enable internet.")

    if not p.exists():
        raise FileNotFoundError(f"DAV2 checkpoint still missing after download: {p}")
    return p


dav2_ckpt = ensure_dav2_checkpoint(CONFIG["dav2_checkpoint_path"])
adapted_ckpt = Path(CONFIG["adapted_checkpoint_path"])
if not adapted_ckpt.exists():
    raise FileNotFoundError(
        f"Adapted checkpoint not found: {adapted_ckpt}. "
        "Upload it as Kaggle input and update CONFIG['adapted_checkpoint_path']."
    )
print("adapted checkpoint:", adapted_ckpt)

In [ ]:
# -----------------------------
# Metrics utilities
# -----------------------------

def to_tensor_rgb(image: Image.Image) -> torch.Tensor:
    arr = np.asarray(image.convert("RGB"), dtype=np.float32) / 255.0
    return torch.from_numpy(arr).permute(2, 0, 1)


def resize_depth_like(depth: torch.Tensor, h: int, w: int) -> torch.Tensor:
    if depth.ndim == 2:
        depth = depth.unsqueeze(0).unsqueeze(0)
    elif depth.ndim == 3:
        depth = depth.unsqueeze(0)
    out = F.interpolate(depth.float(), size=(h, w), mode="bilinear", align_corners=False)
    return out[0, 0]


def valid_mask(depth: np.ndarray) -> np.ndarray:
    return np.isfinite(depth) & (depth > 1e-6)


def median_scale(pred: np.ndarray, gt: np.ndarray, mask: np.ndarray) -> np.ndarray:
    pv = pred[mask]
    gv = gt[mask]
    if pv.size == 0:
        return pred
    s = np.median(gv) / max(np.median(pv), 1e-6)
    return pred * s


def metric_abs_rel(pred: np.ndarray, gt: np.ndarray, mask: np.ndarray) -> float:
    v = np.abs(pred[mask] - gt[mask]) / np.clip(gt[mask], 1e-6, None)
    return float(np.mean(v)) if v.size else np.nan


def metric_rmse(pred: np.ndarray, gt: np.ndarray, mask: np.ndarray) -> float:
    v = (pred[mask] - gt[mask]) ** 2
    return float(np.sqrt(np.mean(v))) if v.size else np.nan


def metric_delta1(pred: np.ndarray, gt: np.ndarray, mask: np.ndarray) -> float:
    p = np.clip(pred[mask], 1e-6, None)
    g = np.clip(gt[mask], 1e-6, None)
    ratio = np.maximum(p / g, g / p)
    return float(np.mean(ratio < 1.25)) if ratio.size else np.nan


def eval_depth_metrics(pred: np.ndarray, gt: np.ndarray) -> dict[str, float]:
    mask = valid_mask(gt)
    if mask.sum() == 0:
        return {
            "abs_rel": np.nan,
            "rmse": np.nan,
            "delta1": np.nan,
            "ms_abs_rel": np.nan,
            "ms_rmse": np.nan,
            "ms_delta1": np.nan,
            "valid_ratio": 0.0,
        }

    raw = {
        "abs_rel": metric_abs_rel(pred, gt, mask),
        "rmse": metric_rmse(pred, gt, mask),
        "delta1": metric_delta1(pred, gt, mask),
    }

    pred_ms = median_scale(pred, gt, mask)
    ms = {
        "ms_abs_rel": metric_abs_rel(pred_ms, gt, mask),
        "ms_rmse": metric_rmse(pred_ms, gt, mask),
        "ms_delta1": metric_delta1(pred_ms, gt, mask),
    }

    return {**raw, **ms, "valid_ratio": float(mask.mean())}


def colorize_depth(depth: np.ndarray) -> np.ndarray:
    d = depth.copy()
    m = np.isfinite(d)
    if m.sum() == 0:
        return np.zeros((depth.shape[0], depth.shape[1], 3), dtype=np.uint8)
    lo, hi = np.percentile(d[m], [2, 98])
    d = np.clip((d - lo) / max(hi - lo, 1e-6), 0, 1)
    c = plt.get_cmap("magma")(d)[..., :3]
    return (c * 255).astype(np.uint8)

In [ ]:
# -----------------------------
# Estimators
# -----------------------------
class BaseEstimator:
    def predict(self, image: Image.Image) -> np.ndarray:
        raise NotImplementedError


class EagleVisionAdaptedEstimator(BaseEstimator):
    def __init__(self):
        depth_model = DepthAnythingWithAdapter(
            mode=CONFIG["depth_mode"],
            encoder=CONFIG["encoder"],
            profile=CONFIG["profile"],
            checkpoint_path=dav2_ckpt,
            freeze_backbone=True,
            adapter_hidden_channels=CONFIG["adapter_hidden_channels"],
            normalize_backbone_input=CONFIG["normalize_backbone_input"],
        ).to(DEVICE).eval()
        self.model = RoundTripDepthNVS(depth_model).to(DEVICE).eval()
        load_checkpoint(adapted_ckpt, self.model)

    @torch.no_grad()
    def predict(self, image: Image.Image) -> np.ndarray:
        x = to_tensor_rgb(image).unsqueeze(0).to(DEVICE)
        d = self.model.depth_model(x)["adapted_depth"][0, 0]
        return d.detach().cpu().numpy().astype(np.float32)


class EagleVisionBaseEstimator(BaseEstimator):
    def __init__(self):
        self.model = DepthAnythingWithAdapter(
            mode=CONFIG["depth_mode"],
            encoder=CONFIG["encoder"],
            profile=CONFIG["profile"],
            checkpoint_path=dav2_ckpt,
            freeze_backbone=True,
            adapter_hidden_channels=CONFIG["adapter_hidden_channels"],
            normalize_backbone_input=CONFIG["normalize_backbone_input"],
        ).to(DEVICE).eval()

    @torch.no_grad()
    def predict(self, image: Image.Image) -> np.ndarray:
        x = to_tensor_rgb(image).unsqueeze(0).to(DEVICE)
        d = self.model(x)["base_depth"][0, 0]
        return d.detach().cpu().numpy().astype(np.float32)


class HFPipelineEstimator(BaseEstimator):
    def __init__(self, model_id: str):
        self.model_id = model_id
        self.pipe = pipeline("depth-estimation", model=model_id, device=0 if DEVICE.type == "cuda" else -1)

    def predict(self, image: Image.Image) -> np.ndarray:
        out = self.pipe(image)
        if "predicted_depth" in out:
            d = out["predicted_depth"]
            if torch.is_tensor(d):
                return d.detach().cpu().numpy().astype(np.float32)
            return np.asarray(d, dtype=np.float32)
        d = out["depth"]
        return np.asarray(d, dtype=np.float32)


class MiDaSEstimator(BaseEstimator):
    def __init__(self, model_type: str = "DPT_Large"):
        self.model = torch.hub.load("intel-isl/MiDaS", model_type).to(DEVICE).eval()
        transforms = torch.hub.load("intel-isl/MiDaS", "transforms")
        self.transform = transforms.dpt_transform if "DPT" in model_type else transforms.small_transform

    @torch.no_grad()
    def predict(self, image: Image.Image) -> np.ndarray:
        arr = np.asarray(image.convert("RGB"))
        inp = self.transform(arr).to(DEVICE)
        pred = self.model(inp)
        pred = F.interpolate(pred.unsqueeze(1), size=arr.shape[:2], mode="bicubic", align_corners=False).squeeze()
        return pred.detach().cpu().numpy().astype(np.float32)


def build_estimator(spec: dict[str, Any]) -> BaseEstimator:
    k = spec["kind"]
    if k == "eaglevision_adapted":
        return EagleVisionAdaptedEstimator()
    if k == "eaglevision_base":
        return EagleVisionBaseEstimator()
    if k == "hf_pipeline":
        return HFPipelineEstimator(spec["hf_model"])
    if k == "torchhub_midas":
        return MiDaSEstimator(spec.get("model_type", "DPT_Large"))
    raise ValueError(f"Unknown kind: {k}")

In [ ]:
# -----------------------------
# Dataset loaders
# -----------------------------

def load_hf_nyu(split: str, max_samples: int):
    ds = load_dataset("sayakpaul/nyu_depth_v2", split=split)
    n = min(len(ds), max_samples)
    rows = []
    for i in range(n):
        ex = ds[i]
        rows.append({
            "sample_id": f"nyu_{i:06d}",
            "image": ex["image"].convert("RGB"),
            "depth": np.asarray(ex["depth_map"], dtype=np.float32),
        })
    return rows


def load_local_scannet_style(root: Path, max_samples: int):
    rows = []
    if not root.exists():
        return rows

    scenes = sorted([p for p in root.glob("*") if p.is_dir()])
    for scene in scenes:
        # Try common layouts
        color_dirs = [scene / "color", scene / "rgb", scene / "images"]
        depth_dirs = [scene / "depth", scene / "depths"]
        cdir = next((d for d in color_dirs if d.exists()), None)
        ddir = next((d for d in depth_dirs if d.exists()), None)
        if cdir is None or ddir is None:
            continue

        colors = sorted(cdir.glob("*.jpg")) + sorted(cdir.glob("*.png"))
        for cp in colors:
            dp_png = ddir / (cp.stem + ".png")
            dp_npy = ddir / (cp.stem + ".npy")
            if dp_png.exists():
                depth = np.asarray(Image.open(dp_png), dtype=np.float32)
                # Heuristic: if values are large, assume mm and convert to meters
                if np.nanmax(depth) > 100:
                    depth = depth / 1000.0
            elif dp_npy.exists():
                depth = np.load(dp_npy).astype(np.float32)
            else:
                continue

            rows.append({
                "sample_id": f"{scene.name}_{cp.stem}",
                "image": Image.open(cp).convert("RGB"),
                "depth": depth,
            })

            if len(rows) >= max_samples:
                return rows

    return rows


def load_folder_pairs(rgb_glob: str, depth_glob: str, depth_scale: float, max_samples: int):
    rgbs = sorted(Path("/").glob(rgb_glob[1:])) if rgb_glob.startswith("/") else sorted((REPO_DIR).glob(rgb_glob))
    deps = sorted(Path("/").glob(depth_glob[1:])) if depth_glob.startswith("/") else sorted((REPO_DIR).glob(depth_glob))
    n = min(len(rgbs), len(deps), max_samples)
    rows = []
    for i in range(n):
        rows.append({
            "sample_id": rgbs[i].stem,
            "image": Image.open(rgbs[i]).convert("RGB"),
            "depth": np.asarray(Image.open(deps[i]), dtype=np.float32) / float(depth_scale),
        })
    return rows


def load_dataset_rows(spec: dict[str, Any], max_samples: int):
    t = spec["type"]
    if t == "hf_nyu":
        return load_hf_nyu(spec.get("split", "validation"), max_samples)
    if t == "local_scannet_style":
        return load_local_scannet_style(Path(spec["root"]), max_samples)
    if t == "folder_pairs":
        return load_folder_pairs(spec["rgb_glob"], spec["depth_glob"], spec.get("depth_scale", 1000.0), max_samples)
    raise ValueError(f"Unknown dataset type: {t}")

In [ ]:
# -----------------------------
# Run benchmark inference
# -----------------------------
enabled_models = [m for m in CONFIG["models"] if m.get("enabled", True)]
enabled_datasets = [d for d in CONFIG["datasets"] if d.get("enabled", True)]

print("Enabled models:")
for m in enabled_models:
    print(" -", m["id"])
print("Enabled datasets:")
for d in enabled_datasets:
    print(" -", d["name"])

estimators = {}
for m in enabled_models:
    print("Loading model:", m["id"])
    estimators[m["id"]] = build_estimator(m)

rows = []
qual_rows = []

for dspec in enabled_datasets:
    dname = dspec["name"]
    print("
=== Dataset:", dname, "===")
    samples = load_dataset_rows(dspec, MAX_SAMPLES)
    print("samples:", len(samples))
    if not samples:
        continue

    qual_idx = set(np.linspace(0, len(samples)-1, min(CONFIG["num_qualitative_pairs"], len(samples))).astype(int).tolist())

    for i, s in enumerate(tqdm(samples, desc=dname)):
        image = s["image"]
        gt = s["depth"].astype(np.float32)
        h, w = gt.shape[:2]

        preds = {}
        for mid, est in estimators.items():
            pred = est.predict(image)
            if pred.ndim == 3:
                pred = pred.squeeze()
            pred = resize_depth_like(torch.from_numpy(pred), h, w).cpu().numpy().astype(np.float32)
            preds[mid] = pred

            metrics = eval_depth_metrics(pred, gt)
            rows.append({
                "dataset": dname,
                "sample_id": s["sample_id"],
                "model": mid,
                **metrics,
            })

        if i in qual_idx:
            qual_rows.append({
                "dataset": dname,
                "sample_id": s["sample_id"],
                "image": np.asarray(image),
                "gt": gt,
                "preds": preds,
            })

per_sample = pd.DataFrame(rows)
per_sample_path = OUT_DIR / "metrics" / "per_sample_metrics.csv"
per_sample.to_csv(per_sample_path, index=False)
print("saved:", per_sample_path)
print(per_sample.head())

In [ ]:
# -----------------------------
# Aggregate + "improved" boolean vs our model
# -----------------------------
if len(per_sample) == 0:
    raise RuntimeError("No results found. Check dataset/model config.")

metric_cols = ["abs_rel", "rmse", "delta1", "ms_abs_rel", "ms_rmse", "ms_delta1", "valid_ratio"]
summary = (
    per_sample.groupby(["dataset", "model"], as_index=False)[metric_cols]
    .mean(numeric_only=True)
)
summary_path = OUT_DIR / "metrics" / "summary_by_dataset_model.csv"
summary.to_csv(summary_path, index=False)
print("saved:", summary_path)

OURS_ID = "ours_adapted"
higher_better = {"delta1", "ms_delta1", "valid_ratio"}

comp_rows = []
for dname in sorted(summary["dataset"].unique()):
    ds = summary[summary["dataset"] == dname].copy()
    ours = ds[ds["model"] == OURS_ID]
    if len(ours) == 0:
        continue
    ours = ours.iloc[0]

    for _, r in ds.iterrows():
        if r["model"] == OURS_ID:
            continue
        for metric in metric_cols:
            ours_val = float(ours[metric])
            other_val = float(r[metric])
            if metric in higher_better:
                improved = bool(ours_val > other_val)
                delta = ours_val - other_val
            else:
                improved = bool(ours_val < other_val)
                delta = other_val - ours_val  # positive means ours better for lower-is-better metrics

            comp_rows.append({
                "dataset": dname,
                "baseline_model": r["model"],
                "metric": metric,
                "ours_value": ours_val,
                "baseline_value": other_val,
                "delta_ours_minus_baseline": ours_val - other_val,
                "delta_positive_means_ours_better": delta,
                "improved": improved,
            })

vs_ours = pd.DataFrame(comp_rows)
vs_ours_path = OUT_DIR / "metrics" / "vs_ours_metric_comparison.csv"
vs_ours.to_csv(vs_ours_path, index=False)
print("saved:", vs_ours_path)
print(vs_ours.head(20))

In [ ]:
# -----------------------------
# Separate tables/plots per baseline vs our model
# -----------------------------
plot_dir = OUT_DIR / "plots" / "per_model"
table_dir = OUT_DIR / "tables" / "per_model"
plot_dir.mkdir(parents=True, exist_ok=True)
table_dir.mkdir(parents=True, exist_ok=True)

if len(vs_ours) == 0:
    print("No vs-ours rows to plot.")
else:
    metrics_focus = ["ms_abs_rel", "ms_rmse", "ms_delta1", "abs_rel", "rmse", "delta1"]

    for (dname, bmodel), g in vs_ours.groupby(["dataset", "baseline_model"]):
        g = g[g["metric"].isin(metrics_focus)].copy()
        g = g.sort_values("metric")

        tpath = table_dir / f"{dname}__{bmodel}__vs_ours.csv"
        g.to_csv(tpath, index=False)

        fig, ax = plt.subplots(figsize=(10, 4.8))
        x = np.arange(len(g))
        vals = g["delta_positive_means_ours_better"].values
        colors = ["#1b9e77" if imp else "#d95f02" for imp in g["improved"].values]
        ax.bar(x, vals, color=colors)
        ax.axhline(0.0, color="black", linewidth=1)
        ax.set_xticks(x)
        ax.set_xticklabels(g["metric"].tolist(), rotation=30, ha="right")
        ax.set_ylabel("Positive means ours better")
        ax.set_title(f"{dname} | ours vs {bmodel}")
        ax.grid(axis="y", alpha=0.25)
        fig.tight_layout()
        ppath = plot_dir / f"{dname}__{bmodel}__vs_ours.png"
        fig.savefig(ppath, dpi=150)
        plt.close(fig)

print("saved per-model tables:", table_dir)
print("saved per-model plots:", plot_dir)

In [ ]:
# -----------------------------
# Qualitative: separate panels (ours vs one baseline at a time)
# -----------------------------
qual_root = OUT_DIR / "qualitative"
qual_root.mkdir(parents=True, exist_ok=True)

baseline_ids = [m["id"] for m in enabled_models if m["id"] != "ours_adapted"]

for bmodel in baseline_ids:
    model_dir = qual_root / f"ours_vs_{bmodel}"
    model_dir.mkdir(parents=True, exist_ok=True)

    for i, q in enumerate(qual_rows):
        if "ours_adapted" not in q["preds"] or bmodel not in q["preds"]:
            continue

        fig, axes = plt.subplots(1, 4, figsize=(16, 4))
        axes[0].imshow(q["image"])
        axes[0].set_title("RGB")
        axes[0].axis("off")

        axes[1].imshow(colorize_depth(q["gt"]))
        axes[1].set_title("GT")
        axes[1].axis("off")

        axes[2].imshow(colorize_depth(q["preds"]["ours_adapted"]))
        axes[2].set_title("Ours")
        axes[2].axis("off")

        axes[3].imshow(colorize_depth(q["preds"][bmodel]))
        axes[3].set_title(bmodel)
        axes[3].axis("off")

        fig.suptitle(f"{q['dataset']} | {q['sample_id']} | ours vs {bmodel}")
        fig.tight_layout()
        out = model_dir / f"{i:03d}_{q['dataset']}_{q['sample_id']}.png"
        fig.savefig(out, dpi=150, bbox_inches="tight")
        plt.close(fig)

print("saved qualitative panels in:", qual_root)

## How to keep runtime short on Kaggle

- Keep `fast_mode=True`.
- Start with 2-3 baselines enabled, then expand.
- Use one dataset first (e.g., NYU validation), then add others.
- Keep MiDaS disabled unless needed.

## Main outputs

- `metrics/per_sample_metrics.csv`
- `metrics/summary_by_dataset_model.csv`
- `metrics/vs_ours_metric_comparison.csv` (includes `improved` boolean)
- `tables/per_model/*.csv` (separate table per baseline vs ours)
- `plots/per_model/*.png` (separate plot per baseline vs ours)
- `qualitative/ours_vs_<baseline>/*.png` (separate qualitative panels)